# CALIDAD DE DATOS

## IMPORTAR PAQUETES

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

#RUTAS
# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


### Uso de utils.py

In [36]:
import sys
from pathlib import Path
# Subir de 03_notebooks a 03_RIESGOS
ROOT = Path.cwd().parent
# Añadir 04_scripts a las rutas de búsqueda
sys.path.insert(0, str(ROOT / "04_scripts"))



## IMPORTAR LOS DATOS

In [3]:
df = pd.read_pickle(ENTRENAMIENTO / 'train.pkl')


## VISIÓN GENERAL

In [4]:
df.head()

,id_cliente,empleo,antigüedad_empleo,ingresos,ingresos_verificados,rating,dti,vivienda,num_hipotecas,num_lineas_credito,...,id_prestamo,descripcion,finalidad,principal,tipo_interes,num_cuotas,imp_cuota,imp_amortizado,estado,imp_recuperado
1,4798121,"Target Promotions and Marketing,Inc",10+ years,65000.000,Not Verified,D,25.400,RENT,1.000,15.000,...,NaN,NaN,debt_consolidation,10000.000,17.770,36 months,360.380,6362.960,Charged Off,0.000
2,46641215,Banker,5 years,135000.000,Verified,A,14.680,RENT,0.000,19.000,...,NaN,NaN,debt_consolidation,24000.000,6.390,36 months,734.380,24000.000,Fully Paid,0.000
3,87998444,executive director,9 years,188000.000,Source Verified,B,11.690,MORTGAGE,3.000,15.000,...,NaN,NaN,credit_card,27000.000,8.990,60 months,560.350,12443.000,Current,0.000
5,131289518,NaN,10+ years,72777.000,Source Verified,D,24.740,MORTGAGE,2.000,10.000,...,NaN,NaN,home_improvement,25000.000,21.850,60 months,688.350,2811.270,Current,0.000
7,103448519,Web site administrator,10+ years,99800.000,Not Verified,A,9.690,MORTGAGE,0.000,8.000,...,NaN,NaN,credit_card,9600.000,5.320,36 months,289.110,6228.920,Current,0.000


## TIPOS DE DATOS

### Identificación

In [11]:
df.reset_index(drop=True,inplace=True)

In [12]:
df.head().T

,0,1,2,3,4
id_cliente,4798121,46641215,87998444,131289518,103448519
empleo,"Target Promotions and Marketing,Inc",Banker,executive director,NaN,Web site administrator
antigüedad_empleo,10+ years,5 years,9 years,10+ years,10+ years
ingresos,65000.000,135000.000,188000.000,72777.000,99800.000
ingresos_verificados,Not Verified,Verified,Source Verified,Source Verified,Not Verified
rating,D,A,B,D,A
dti,25.400,14.680,11.690,24.740,9.690
vivienda,RENT,RENT,MORTGAGE,MORTGAGE,MORTGAGE
num_hipotecas,1.000,0.000,3.000,2.000,0.000
num_lineas_credito,15.000,19.000,15.000,10.000,8.000


Conclusiones:

* num_hipotecas, num_lineas_credito, num_cancelaciones_12meses, num_derogatorios, num_meses_desde_ult_retraso deberían ser int
* pero vemos que hay nulos, así que no podemos convertirlas hasta que los imputemos

## VALORES ÚNICOS

### Identificación

In [14]:
df.nunique().sort_values()

id_prestamo                         0
num_cuotas                          2
ingresos_verificados                3
vivienda                            6
num_cancelaciones_12meses           7
rating                              7
estado                              9
antigüedad_empleo                  11
finalidad                          14
num_derogatorios                   20
num_hipotecas                      25
num_lineas_credito                 65
num_meses_desde_ult_retraso       132
porc_tarjetas_75p                 173
tipo_interes                      580
porc_uso_revolving               1101
principal                        1463
dti                              5297
descripcion                      7770
imp_recuperado                  10599
ingresos                        11675
imp_cuota                       33539
empleo                          54479
imp_amortizado                  57851
id_cliente                     140000
dtype: int64

Conclusiones:

* id_prestamo con menos de 2 valores --> eliminar
* empleo y descripcion tienen tantos valores distintos que parecen texto libre, escrito por el gestor tal como lo dijo el cliente, sin lista desplegable en el sistema. Se descartan en esta fase. Normalizarlos sería un mini proyecto aparte: limpiar el texto y agrupar los empleos en categorías.

### Corrección

In [15]:
df.drop(columns='id_prestamo', inplace=True)

## DUPLICADOS

In [17]:
df.duplicated().sum()

np.int64(0)

### Identificación

## SEPARAR NUMÉRICAS Y CATEGÓRICAS

### Categóricas

In [18]:
cat = df.select_dtypes(include='string')

### Numéricas

In [19]:
num = df.select_dtypes(include='number')

## GESTIÓN DE CATEGÓRICAS

### Nulos

In [21]:
cat.isnull().sum()

empleo                   10310
antigüedad_empleo         9052
ingresos_verificados         0
rating                       0
vivienda                     0
descripcion             132208
finalidad                    0
num_cuotas                   0
estado                       0
dtype: int64

In [25]:
cat['antigüedad_empleo'].value_counts(dropna=False)

antigüedad_empleo
10+ years    46117
2 years      12523
< 1 year     11884
3 years      11215
1 year        9124
NaN           9052
5 years       8728
4 years       8547
6 years       6334
7 years       5800
8 years       5739
9 years       4937
Name: count, dtype: int64

Conclusiones:

* eliminar descripción y empleo
* imputar nulos de antigüedad_empleo por valor 'desconocido'

#### Corrección

In [ ]:
#Como se habia decidido, se borrara empleo y descripcion
cat.drop(columns=['empleo',], inplace 'descripcion'=True)

In [26]:
cat['antigüedad_empleo'] = cat['antigüedad_empleo'].fillna('Desconocido')

## GESTIÓN DE NUMÉRICAS

### Nulos

#### Identificación

In [27]:
num.isna().sum().sort_values(ascending=False)

num_meses_desde_ult_retraso    71743
porc_tarjetas_75p               4609
num_hipotecas                   3073
porc_uso_revolving               109
dti                              102
num_cancelaciones_12meses          8
num_lineas_credito                 3
num_derogatorios                   3
ingresos                           0
id_cliente                         0
principal                          0
tipo_interes                       0
imp_cuota                          0
imp_amortizado                     0
imp_recuperado                     0
dtype: int64

Veamoslas con más detalle.

In [29]:
num['num_meses_desde_ult_retraso'].value_counts(dropna=False).sort_index()

num_meses_desde_ult_retraso
0.000        173
1.000        447
2.000        631
3.000        787
4.000        963
           ...  
151.000        1
159.000        1
180.000        1
195.000        1
NaN        71743
Name: count, Length: 133, dtype: int64

Hay ceros, pero parecen pocos para una variable que implica retrasos. Así que vamos a pensar que los nulos significan que no ha habido retrasos. Y por tanto imputar por ceros.

In [30]:
num['porc_tarjetas_75p'].value_counts(dropna=False).sort_index()

porc_tarjetas_75p
0.000      36973
0.170          1
0.250          2
0.330          1
0.500          3
           ...  
94.100         2
94.400         3
94.700         1
100.000    23150
NaN         4609
Name: count, Length: 174, dtype: int64

La categoría mayoritaria es cero. Imputaremos por cero. Los clientes con NaN no tienen tarjetas.

In [31]:
num['num_hipotecas'].value_counts(dropna=False).sort_index()

num_hipotecas
0.000     57496
1.000     24540
2.000     20213
3.000     14497
4.000      9208
5.000      5277
6.000      2826
7.000      1425
8.000       688
9.000       345
10.000      186
11.000       91
12.000       43
13.000       32
14.000       21
15.000       12
16.000        6
17.000        7
18.000        4
19.000        3
20.000        2
21.000        2
23.000        1
25.000        1
32.000        1
NaN        3073
Name: count, dtype: int64

La categoría mayoritaria es cero. Imputaremos por cero.

#### Corrección

In [32]:
num.fillna(0, inplace=True)

,id_cliente,ingresos,dti,num_hipotecas,num_lineas_credito,porc_tarjetas_75p,porc_uso_revolving,num_cancelaciones_12meses,num_derogatorios,num_meses_desde_ult_retraso,principal,tipo_interes,imp_cuota,imp_amortizado,imp_recuperado
0,4798121,65000.000,25.400,1.000,15.000,85.700,86.200,0.000,0.000,0.000,10000.000,17.770,360.380,6362.960,0.000
1,46641215,135000.000,14.680,0.000,19.000,0.000,16.300,0.000,0.000,0.000,24000.000,6.390,734.380,24000.000,0.000
2,87998444,188000.000,11.690,3.000,15.000,0.000,3.300,0.000,0.000,0.000,27000.000,8.990,560.350,12443.000,0.000
3,131289518,72777.000,24.740,2.000,10.000,33.300,63.200,0.000,0.000,0.000,25000.000,21.850,688.350,2811.270,0.000
4,103448519,99800.000,9.690,0.000,8.000,0.000,25.300,0.000,0.000,28.000,9600.000,5.320,289.110,6228.920,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139995,119242742,20000.000,8.020,0.000,5.000,66.700,33.200,0.000,0.000,0.000,2500.000,12.620,83.780,996.060,0.000
139996,51876926,42000.000,20.850,6.000,9.000,100.000,81.600,0.000,0.000,13.000,8000.000,12.290,266.830,8000.000,0.000
139997,121031962,111697.000,16.630,2.000,10.000,50.000,26.300,0.000,1.000,6.000,10000.000,9.440,320.050,4388.510,0.000
139998,53664762,168000.000,4.690,0.000,8.000,0.000,52.400,0.000,2.000,0.000,30050.000,18.250,767.170,2964.440,2750.880


Una vez ya imputados los ceros podemos cambiar los tipos a int.

In [33]:
# num_hipotecas, num_lineas_credito, num_cancelaciones_12meses, num_derogatorios, num_meses_desde_ult_retraso, pasan a ser int
num['num_hipotecas'] = num['num_hipotecas'].astype(int)
num['num_lineas_credito'] = num['num_lineas_credito'].astype(int)
num['num_cancelaciones_12meses'] = num['num_cancelaciones_12meses'].astype(int)
num['num_derogatorios'] = num['num_derogatorios'].astype(int)
num['num_meses_desde_ult_retraso'] = num['num_meses_desde_ult_retraso'].astype(int)

### Atípicos

In [44]:
num.describe().T

,count,mean,std,min,25%,50%,75%,max
id_cliente,140000.000,80347672.702,44899606.621,65640.000,45314414.250,84517226.500,122303273.500,145637006.000
ingresos,140000.000,78292.112,87857.290,0.000,46100.000,65000.000,93000.000,10999200.000
dti,140000.000,18.781,14.413,-1.000,11.900,17.780,24.420,999.000
num_hipotecas,140000.000,1.514,1.881,0.000,0.000,1.000,2.000,32.000
num_lineas_credito,140000.000,11.615,5.649,0.000,8.000,11.000,14.000,91.000
porc_tarjetas_75p,140000.000,41.063,36.419,0.000,0.000,33.300,66.700,100.000
porc_uso_revolving,140000.000,50.319,24.703,0.000,31.700,50.300,69.200,366.600
num_cancelaciones_12meses,140000.000,0.009,0.108,0.000,0.000,0.000,0.000,7.000
num_derogatorios,140000.000,0.199,0.582,0.000,0.000,0.000,0.000,46.000
num_meses_desde_ult_retraso,140000.000,16.831,23.057,0.000,0.000,0.000,31.000,195.000


In [45]:
num.loc[num['ingresos']==0]

,id_cliente,ingresos,dti,num_hipotecas,num_lineas_credito,porc_tarjetas_75p,porc_uso_revolving,num_cancelaciones_12meses,num_derogatorios,num_meses_desde_ult_retraso,principal,tipo_interes,imp_cuota,imp_amortizado,imp_recuperado
4257,139600981,0.000,0.000,0,13,100.000,67.200,0,0,0,15000.000,27.270,460.460,759.540,0.000
5567,130093563,0.000,0.000,0,12,0.000,16.100,0,1,0,7000.000,6.710,215.220,2179.150,0.000
6418,139687198,0.000,0.000,4,13,0.000,20.800,0,0,0,20000.000,6.110,609.440,2564.010,0.000
10387,116575508,0.000,0.000,1,8,80.000,78.600,0,0,0,10000.000,10.420,324.650,10000.000,0.000
11444,143653697,0.000,0.000,3,23,45.500,52.800,0,0,0,33175.000,12.980,754.500,1608.490,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136490,143193530,0.000,0.000,1,4,50.000,68.800,0,0,0,10000.000,11.060,217.730,509.000,0.000
136792,130406634,0.000,0.000,0,12,66.700,50.700,0,1,0,12000.000,13.580,276.620,1639.720,0.000
137309,126292076,0.000,0.000,1,10,33.300,24.000,0,1,68,17000.000,16.020,413.590,2411.670,0.000
138195,142167168,0.000,0.000,0,6,50.000,31.200,0,0,0,8800.000,10.080,284.290,1069.670,0.000


#### Por desviación típica

##### Identificación

In [38]:
# Importar la función
from utils import atipicos_desv_tip

In [ ]:
def atipicos_desv_tip(variable, x=4):
    variable = variable.dropna()          # los nulos, fuera por ahora

    media = np.mean(variable)
    sd = np.std(variable)
    umbral = sd * x

    lim_inf = media - umbral
    lim_sup = media + umbral

    # Índices de los que caen fuera de los límites (con bucle tradicional)
    indices = []

    # .items() recorre la Serie devolviendo pares (índice, valor)
    for indice, valor in variable.items():
        if valor < lim_inf or valor > lim_sup:
            indices.append(indice)

    return indices


In [67]:
#funcion que cuenta el numero de atipicos
def contar_atipicos(df, variable,num_desv_tip):
    atipicos = atipicos_desv_tip(df[variable], num_desv_tip)
    return (df.loc[atipicos,variable].value_counts().sort_index())

Variables a aplicar

In [71]:
var_atipicas_dt = ['ingresos','principal','imp_cuota','imp_recuperado']

Obtener los 10 primeros atipicos ordenados en cada variable

In [75]:
#Obtiene los 10 primeros atípicos por variable
for variable in var_atipicas_dt:
    print('\n' + variable +':\n')
    conteo = contar_atipicos(num, variable, 3.7)
    print(conteo.sort_index().head(15))


ingresos:

ingresos
404000.000     1
405000.000     2
406000.000     1
410000.000     4
410396.000     1
415000.000     4
417000.000     1
418000.000     1
420000.000     6
422000.000     1
422550.000     1
423000.000     1
424000.000     1
425000.000    11
427000.000     1
Name: count, dtype: int64

principal:

Series([], Name: count, dtype: int64)

imp_cuota:

imp_cuota
1443.760    2
1445.500    1
1445.900    3
1447.310    2
1451.140    1
1461.380    1
1466.040    2
1471.620    1
1476.050    1
1479.870    2
1482.070    1
1485.630    1
1486.340    1
1486.550    1
1489.070    1
Name: count, dtype: int64

imp_recuperado:

imp_recuperado
3006.900    1
3006.980    1
3008.540    1
3008.650    1
3010.090    1
3010.960    1
3011.020    1
3013.470    1
3015.420    1
3015.830    1
3015.850    1
3017.570    1
3018.100    1
3018.190    1
3019.090    1
Name: count, dtype: int64


Revisamos con más detalle la variable ingresos.

In [76]:
num['ingresos'].describe()

count     140000.000
mean       78292.112
std        87857.290
min            0.000
25%        46100.000
50%        65000.000
75%        93000.000
max     10999200.000
Name: ingresos, dtype: float64

##### Corrección

Eliminar los registros con ingresos superiores a 400.000$, buen cap considerando los datos

*Cuidado: Si se eliminan registros hay que eliminarlos tanto de cat como de num*

###### Id de registros a eliminar

In [77]:
a_eliminar = num.loc[num['ingresos'] > 400000].index.values

###### Aplicar

In [78]:
cat = cat[~cat.index.isin(a_eliminar)]
num = num[~num.index.isin(a_eliminar)]

## GUARDAR DATASETS TRAS CALIDAD DE DATOS

Se guardan los avances en df, cat y num con un sufijo para poder guardar o recuperar avances del proyecto.

En formato pickle para no perder las modificaciones de metadatos.

In [79]:
df.to_pickle(ENTRENAMIENTO/'train_calidad.pkl')
cat.to_pickle(ENTRENAMIENTO/'cat_calidad.pkl')
num.to_pickle(ENTRENAMIENTO/'num_calidad.pkl')